# 🩺 Predictor de Diabetes — Árbol de Decisión
### Dataset: Pima Indians Diabetes (UCI Machine Learning Repository)
#### 768 registros · Mujeres mayores de 21 años · Variables clínicas reales

La diabetes es una de las principales causas de mortalidad en el mundo.
Este notebook aplica Machine Learning para predecir si un paciente
tiene diabetes según indicadores clínicos como glucosa, IMC y edad.



In [2]:
import pandas as pd
ruta_file ='/kaggle/input/datasets/sebastianmamani2004/diabetes-csv/diabetes.csv'
diabetes_data = pd.read_csv(ruta_file)



## 🧹 Limpieza de datos
Eliminamos filas con valores faltantes para asegurar
que el modelo entrene con datos completos y confiables.

In [3]:
diabetes_data = diabetes_data.dropna(axis=0)

In [4]:
diabetes_data.columns
diabetes_data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 🎯 Selección de variables
**Objetivo (y):** `Outcome` — 1 si tiene diabetes, 0 si no tiene.

**Features (X):** Variables clínicas con mayor relevancia médica:
- `Glucose` → indicador principal de diabetes
- `BMI` → obesidad como factor de riesgo directo
- `Age` → el riesgo aumenta con la edad
- `DiabetesPedigreeFunction` → historial familiar
- `BloodPressure` → presión arterial como indicador secundario

## 🌳 Entrenamiento — Árbol de Decisión
Usamos `DecisionTreeClassifier` de scikit-learn.
El modelo aprende patrones clínicos de los datos y construye
un árbol de decisiones para clasificar nuevos pacientes.

In [5]:
y = diabetes_data.Outcome

In [6]:
features = ['Glucose', 'BMI', 'Age', 'DiabetesPedigreeFunction', 'BloodPressure']
X = diabetes_data[features]

## 🌳 Entrenamiento — Árbol de Decisión
Usamos `DecisionTreeClassifier` de scikit-learn.
El modelo aprende patrones clínicos de los datos y construye
un árbol de decisiones para clasificar nuevos pacientes.

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

diabetes_model = DecisionTreeClassifier(random_state=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

diabetes_model.fit(X_train, y_train)



DecisionTreeClassifier(random_state=1)

## 🔮 Predicciones vs Valores Reales
Comparamos las predicciones del modelo contra los diagnósticos reales
de las primeras 5 pacientes del dataset.

In [8]:
print("valores")
print(X_test.head())
print('predicciones')
print(diabetes_model.predict(X_test.head()))
print('Comparación')
print(y_test.head().tolist())



valores
     Glucose   BMI  Age  DiabetesPedigreeFunction  BloodPressure
285      136  26.0   51                     0.647             74
101      151  26.1   22                     0.179             60
581      109  25.0   27                     0.206             60
352       61  34.4   46                     0.243             82
726      116  36.1   25                     0.496             78
predicciones
[0 0 0 0 0]
Comparación
[0, 0, 0, 0, 0]


📊 Evaluación del Modelo
Evaluamos el modelo con datos que nunca vio durante el entrenamiento (X_test). Usamos tres métricas:

MAE: error promedio entre predicción y valor real
Accuracy: porcentaje de aciertos general
Classification Report: precisión detallada por clase (0 y 1)

In [12]:
val_predic=diabetes_model.predict(X_test)
print(val_predic)
from sklearn.metrics import mean_absolute_error
val_mae = mean_absolute_error(val_predic, y_test)
print(val_mae)
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

print(accuracy_score(y_test, val_predic))
print(classification_report(y_test, val_predic))

[0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 1 1 1 0
 0 0 1 0 0 0 1 0 0 1 0 0 0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 1 0 0 0 1 1 1 1 1 0
 0 1 1 1 0 1 1 0 0 0 0 1 0 1 0 1 0 0 1 0 1 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 1 1 1 1 1 0 0 0 0 1 0 0 0 1 0 1 1 0 0 0 1 0 0 1 0 0 1 1 1 0 0
 0 0 0 1 0 0]
0.2857142857142857
0.7142857142857143
              precision    recall  f1-score   support

           0       0.77      0.79      0.78        99
           1       0.60      0.58      0.59        55

    accuracy                           0.71       154
   macro avg       0.69      0.68      0.69       154
weighted avg       0.71      0.71      0.71       154



🔍 Buscando el Árbol Óptimo
Un árbol con demasiadas hojas memoriza los datos (overfitting). Un árbol con muy pocas hojas no aprende suficiente (underfitting).

Probamos distintos valores de max_leaf_nodes para encontrar el balance óptimo entre ambos extremos.

In [14]:
hojas_pruebas = [2,3,4,5,20,30,50,100,150,200,250,400,500]

🌳 Comparación de MAE por número de hojas
Entrenamos el modelo con diferentes límites de hojas y medimos el error en datos de validación para identificar el punto óptimo.

In [15]:
def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeClassifier(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(train_X, train_y)
    val_predictions = model.predict(val_X)
    mae = mean_absolute_error(val_predictions, val_y)
    return mae
for max_leaf_nodes in hojas_pruebas:
  print(get_mae(max_leaf_nodes, X_train, X_test, y_train, y_test))

0.2662337662337662
0.2012987012987013
0.2012987012987013
0.2012987012987013
0.2077922077922078
0.22727272727272727
0.24025974025974026
0.2727272727272727
0.2792207792207792
0.2792207792207792
0.2792207792207792
0.2792207792207792
0.2792207792207792


🔍 Buscando el Árbol Óptimo
Un árbol con demasiadas hojas memoriza los datos (overfitting). Un árbol con muy pocas hojas no aprende suficiente (underfitting).

A diferencia de regresión donde buscamos el menor MAE, en clasificación buscamos el mayor Accuracy.



In [16]:
def get_accuracy(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeClassifier(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(train_X, train_y)
    val_predictions = model.predict(val_X)
    return accuracy_score(val_y, val_predictions)

for max_leaf_nodes in hojas_pruebas:
    print(f"Hojas: {max_leaf_nodes} → Accuracy: {get_accuracy(max_leaf_nodes, X_train, X_test, y_train, y_test):.2%}")

Hojas: 2 → Accuracy: 73.38%
Hojas: 3 → Accuracy: 79.87%
Hojas: 4 → Accuracy: 79.87%
Hojas: 5 → Accuracy: 79.87%
Hojas: 20 → Accuracy: 79.22%
Hojas: 30 → Accuracy: 77.27%
Hojas: 50 → Accuracy: 75.97%
Hojas: 100 → Accuracy: 72.73%
Hojas: 150 → Accuracy: 72.08%
Hojas: 200 → Accuracy: 72.08%
Hojas: 250 → Accuracy: 72.08%
Hojas: 400 → Accuracy: 72.08%
Hojas: 500 → Accuracy: 72.08%


🏆 Resultado
El modelo óptimo tiene 3 hojas con un accuracy de 79.87%.

Con menos hojas → underfitting (demasiado simple)
Con más hojas → overfitting (memoriza en vez de aprender)
3 hojas → balance perfecto para este dataset